<a href="https://colab.research.google.com/github/preeti-bhadauria/Neural-Machine-Translation/blob/main/enco_deco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt

In [6]:
!find /content -name "spa.txt"


/content/spa.txt


In [7]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"


In [8]:
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, cache_dir="datasets",
                               extract=True)

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
text = (Path(path).with_name("spa-eng") / "/content/spa.txt").read_text()

In [10]:
text.replace("¡","").replace("¿","") #replacing bcz textvectorization layer doesnt handle these
pairs = [line.split("\t") for line in text.splitlines()]
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs) #separates the pair in two lists

In [11]:
for i in range(3):
  print(sentences_en[i], "=>", sentences_es[i])

We need to prepare for the worst. => Debemos prepararnos para lo peor.
They told me to wait here. => Me dijeron que esperara aquí.
What will we be doing this time next week? => ¿Qué estaremos haciendo a esta hora la próxima semana?


create two vectorization layer -- one per language

In [12]:
vocab_size = 1000
max_length = 50
text_vec_layer_en = tf.keras.layers.TextVectorization(
    vocab_size,output_sequence_length=max_length)
text_vec_layer_es = tf.keras.layers.TextVectorization(
    vocab_size,output_sequence_length=max_length)
text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es])

inspect first 10 tokens in both vocabularies-- start with padding, unkown,(sos,eos-esp only)

In [40]:
text_vec_layer_en.get_vocabulary()[:10]

['',
 '[UNK]',
 np.str_('the'),
 np.str_('i'),
 np.str_('to'),
 np.str_('you'),
 np.str_('tom'),
 np.str_('a'),
 np.str_('is'),
 np.str_('he')]

In [41]:
text_vec_layer_es.get_vocabulary()[:10]

['',
 '[UNK]',
 np.str_('startofseq'),
 np.str_('endofseq'),
 np.str_('de'),
 np.str_('que'),
 np.str_('a'),
 np.str_('no'),
 np.str_('tom'),
 np.str_('la')]

create training set and validation set

In [13]:
X_train = tf.constant(sentences_en[:100_000])
X_valid = tf.constant(sentences_en[100_000:])
X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:100_000]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[100_000:]])
Y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:100_000]])
Y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[100_000:]])


Build translation model-- use functional API since the model is not sequential-- two text input- one for encoder one for decoder

In [14]:
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

Encode sentences using TextVectorization layer, followed by embedding layer with mask_zero=true

In [15]:
embed_size = 128
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

create the encoder and pass it the embedded input

In [18]:
encoder = tf.keras.layers.LSTM(512, return_state=True)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

In [19]:
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

pass the decoders output through a dense layer with activation function

In [20]:
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(decoder_outputs)


In [21]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath="seq2seq_translation.keras",
    monitor="val_loss",          # best metric for translation
    save_best_only=True,
    verbose=1
)

In [22]:
early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


create keras model, compile & train

In [24]:
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
model.fit((X_train,X_train_dec), Y_train, epochs=6, validation_data=((X_valid, X_valid_dec), Y_valid), callbacks=[checkpoint_cb, early_stop_cb])

Epoch 1/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0724 - loss: 2.2478
Epoch 1: val_loss improved from inf to 1.78182, saving model to seq2seq_translation.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 4615s 1s/step - accuracy: 0.0724 - loss: 2.2477 - val_accuracy: 0.0831 - val_loss: 1.7818
Epoch 2/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0871 - loss: 1.6092
Epoch 2: val_loss improved from 1.78182 to 1.49114, saving model to seq2seq_translation.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 4803s 2s/step - accuracy: 0.0871 - loss: 1.6092 - val_accuracy: 0.0906 - val_loss: 1.4911
Epoch 3/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0960 - loss: 1.2847
Epoch 3: val_loss improved from 1.49114 to 1.36884, saving model to seq2seq_translation.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 4805s 2s/step - accuracy: 0.0960 - loss: 1.2847 - val_accuracy: 0.0938 - val_loss: 1.3688
Epoch 4/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1019 - loss: 1.0830
Epoch 4: val_lo

In [25]:
import tensorflow as tf

model = tf.keras.models.load_model("seq2seq_translation.keras")


In [26]:
def translate(sentence_en):
    translation = ""

    for word_idx in range(max_length):
        # 1️⃣ Vectorize encoder input
        X = text_vec_layer_en([sentence_en])

        # 2️⃣ Vectorize decoder input
        X_dec = text_vec_layer_es(["startofseq " + translation])

        # 3️⃣ Predict
        y_proba = model.predict((X, X_dec), verbose=0)[0, word_idx]

        # 4️⃣ Get predicted token
        predicted_word_id = np.argmax(y_proba)
        predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]

        # 5️⃣ Stop condition
        if predicted_word == "endofseq":
            break

        translation += " " + predicted_word

    return translation.strip()


In [27]:
def translate(sentence_en):
    translation = ""

    for word_idx in range(max_length):
        X = tf.constant([sentence_en])  # ✅ string tensor
        X_dec = tf.constant(["startofseq " + translation])  # ✅ string tensor

        y_proba = model.predict((X, X_dec), verbose=0)[0, word_idx]
        predicted_word_id = np.argmax(y_proba)
        predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]

        if predicted_word == "endofseq":
            break

        translation += " " + predicted_word

    return translation.strip()


In [28]:
translate("I love soccer")

'me gusta el fútbol'

In [29]:
translate("i love soccer and also going the beach")

'me encanta el béisbol y también [UNK]'

In [30]:
encoder = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(512, return_state=True, return_sequences=True))

In [31]:
from tensorflow.keras.layers import Concatenate

encoder_outputs, *encoder_state = encoder(encoder_embeddings)

h = Concatenate(axis=-1)(encoder_state[::2])
c = Concatenate(axis=-1)(encoder_state[1::2])

encoder_state = [h, c]


In [32]:
attention_layer = tf.keras.layers.Attention()
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

In [33]:
translate("i love soccer and also going the beach")

'me encanta el béisbol y también [UNK]'

In [34]:
translate("I love soccer")

'me gusta el fútbol'

In [35]:
translate("I love you")

'te amo'